# EDEN-08 — Climate baselines | v1.0

**Original plan: Phase 2, Notebook 08.** Establish honest reference forecasts for temperature, relative humidity and CO₂ at **15, 30, 60, 180 and 360 minutes**, using the frozen EDEN-07 windows. Compare persistence, a training-only seasonal average, Ridge, and gradient boosting on identical validation rows. This is the climate reference stage before original Notebook 09 physics calibration.

**Kaggle:** CPU, Internet **Off**. Attach only `EDEN-07-Climate-Vision-Alignment-Results-v1.0.zip` or its extracted folder. No GPU, raw images, or extra dataset. Run in order and return `EDEN-08-Climate-Baselines-Results-v1.0.zip`.

**Verified input:** EDEN-07 has 16 verified manifest files, 259,932 aligned rows (129,966 climate observations repeated for two protocols), and 1,273,094 valid window-index rows. Both protocols keep their own splits and training-only scalers. Image coverage is sparse; this notebook establishes climate-only comparisons. Vision effects belong in later matched fusion experiments.

**A necessary plan qualification:** daily seasonal persistence would use the reading at `origin + horizon − 24 hours`, outside the allowed six-hour history. It is explicitly **not run**. A separately named `seasonal_mean_train_only` predicts the average training value at the target UTC clock time. It is a fitted seasonal reference, not yesterday's observation and not a disguised replacement for seasonal persistence.

**No test scoring:** training fits model parameters; development validation chooses Ridge penalties and compares models. Test rows are not used to build features, fit, select, or score models. Earlier prototype test summaries were already viewed, so the project cannot claim a completely untouched test history. Model selection and intervals here are exploratory validation evidence.

**Compute:** full training data, 10 multi-target Ridge searches and 30 fixed-configuration boosted regressors. Models are cached after each protocol/horizon finishes, so rerunning all cells in the same session can reuse completed blocks. This notebook was not executed locally.

API references: [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [histogram gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html). Dataset attribution: [2024 greenhouse challenge](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0.

## Cell 1 — Settings and folders

Persistence repeats the current reading. Ridge learns a weighted sum; boosting combines small decision trees. All learned models use the same recorded history and known calendar information. No future weather or actual future actions enter the inputs.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import gc, hashlib, io, json, zipfile, importlib.metadata
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ROOT = WORK / 'eden08'
CACHE = WORK / 'eden08_cache'
RESULT = WORK / 'EDEN-08-Climate-Baselines-Results-v1.0.zip'
SEED = 42
ALPHAS = [0.1, 1.0, 10.0, 100.0]
HGB_SETTINGS = dict(max_iter=120, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=64,
                    l2_regularization=1.0, max_bins=128, early_stopping=False, random_state=SEED)
BOOTSTRAP_REPEATS = 1000
TARGETS = ['air_temp_c', 'rh_pct', 'co2_ppm']
ERROR_UNITS = ['degree_Celsius', 'percentage_points', 'ppm']
for directory in [ROOT / 'data', ROOT / 'models', ROOT / 'reports', CACHE]:
    directory.mkdir(parents=True, exist_ok=True)

def sha(payload):
    """Fingerprint exact bytes for input lineage and saved models."""
    return hashlib.sha256(payload).hexdigest()

def save_json(path, value):
    """Write portable metadata; reject NaN/infinity in JSON reports."""
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

versions = {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'pyarrow', 'scikit-learn', 'joblib']}
save_json(ROOT / 'reports/runtime.json', versions)
print('CPU; development validation only; no test scoring.')

## Cell 2 — Verify EDEN-07 and its contracts

A manifest is the result receipt. Check its files before trusting table row numbers, window definitions, or fitted scalers. A ZIP or an extracted Kaggle folder is accepted.

In [ ]:
class Source:
    def __init__(self, path, member=None):
        self.path, self.member = Path(path), member
        if member is None:
            self.manifest_bytes = self.path.read_bytes()
            self.base = self.path.parent
        else:
            with zipfile.ZipFile(self.path) as archive:
                self.manifest_bytes = archive.read(member)
            self.prefix = str(PurePosixPath(member).parent)
        self.manifest = json.loads(self.manifest_bytes)

    def read(self, relative):
        # Accept only paths relative to the producer's artifact directory.
        relative = str(relative).replace(chr(92), '/')
        parts = PurePosixPath(relative)
        assert not parts.is_absolute() and '..' not in parts.parts and ':' not in relative
        if self.member is None:
            target = (self.base / relative).resolve()
            assert target.is_relative_to(self.base.resolve())
            return target.read_bytes()
        name = str(PurePosixPath(self.prefix) / parts)
        with zipfile.ZipFile(self.path) as archive:
            return archive.read(name)




candidates = {}

def consider(source):
    """Recognize the exact upstream producer/version, not a guessed folder name."""
    if source.manifest.get('producer') == 'EDEN-07' and str(source.manifest.get('code_version')) == '1.0':
        candidates[sha(source.manifest_bytes)] = source

for path in sorted(INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(path) as bundle:
            names = [n for n in bundle.namelist() if PurePosixPath(n).name == 'manifest.json']
        for name in names:
            consider(Source(path, name))
    except (zipfile.BadZipFile, json.JSONDecodeError, UnicodeDecodeError):
        continue
for path in sorted(INPUT.rglob('manifest.json')):
    try:
        consider(Source(path))
    except (json.JSONDecodeError, UnicodeDecodeError):
        continue
assert len(candidates) == 1, 'Attach exactly one distinct EDEN-07 v1.0 result.'
source = next(iter(candidates.values()))
assert source.manifest['status'] == 'PASS_ALIGNMENT_SPARSE_VISION_TRANSFER_UNVALIDATED'
files = source.manifest['files']
assert files and len(files) == len({f['path'] for f in files})
for item in files:
    payload = source.read(item['path'])
    assert len(payload) == item['size'] and sha(payload) == item['sha256'], item['path']
upstream_hash = sha(source.manifest_bytes)
contracts = {name: json.loads(source.read('data/' + name + '.json'))
             for name in ['windows', 'scaler', 'feature_contract', 'splits']}
window_contract, upstream_scaler = contracts['windows'], contracts['scaler']
assert window_contract['lookback_steps'] == 72 and window_contract['timestep_minutes'] == 5
assert window_contract['horizon_minutes'] == [15, 30, 60, 180, 360]
assert window_contract['target_columns'] == TARGETS
assert upstream_scaler['fitted_on'] == 'training split only'
assert source.manifest['test_scored'] is False
HORIZONS = window_contract['horizon_minutes']
PROTOCOLS = ['primary', 'chronological']
for name in ['windows', 'scaler', 'feature_contract', 'splits']:
    (ROOT / 'data' / ('upstream_' + name + '.json')).write_bytes(source.read('data/' + name + '.json'))
save_json(ROOT / 'reports/input_verification.json', dict(producer='EDEN-07', manifest_sha256=upstream_hash,
    files_verified=len(files), status=source.manifest['status']))
print('Verified EDEN-07:', len(files), 'files')

## Cell 3 — Load development rows and independently check windows

Test rows are excluded before feature construction. We check every timestamp between the start of history and the target, not just the two endpoints. This catches hidden gaps and split crossings.

In [ ]:
history_candidates = contracts['feature_contract']['history_candidates']
columns = ['row_id', 'source_row_id', 'protocol', 'experiment_id', 'compartment_id', 'split', 'ts_utc', 'vision_usable']
columns += history_candidates + [c + '__valid' for c in history_candidates]
table = pd.read_parquet(io.BytesIO(source.read('data/aligned.parquet')), columns=columns)
table['ts_utc'] = pd.to_datetime(table.ts_utc, utc=True).astype('datetime64[ns, UTC]')
assert np.array_equal(table.row_id.to_numpy(), np.arange(len(table)))
windows = pd.read_parquet(io.BytesIO(source.read('data/window_index.parquet')))
all_window_count = len(windows)
windows = windows[windows['split'].isin(['train', 'validation'])].copy().reset_index(drop=True)
assert not windows.duplicated(['protocol', 'origin_row', 'horizon_steps']).any()
for column in ['origin_ts_utc', 'target_ts_utc']:
    windows[column] = pd.to_datetime(windows[column], utc=True).astype('datetime64[ns, UTC]')
start = windows.history_start_row.to_numpy(np.int64)
origin = windows.origin_row.to_numpy(np.int64)
target = windows.target_row.to_numpy(np.int64)
assert (origin - start == 71).all() and np.array_equal(target - origin, windows.horizon_steps.to_numpy())
keys = ['protocol', 'experiment_id', 'compartment_id', 'split']
for key in keys:
    assert np.array_equal(table[key].to_numpy()[start], windows[key].to_numpy())
    assert np.array_equal(table[key].to_numpy()[target], windows[key].to_numpy())
transition = table[keys].ne(table[keys].shift()).any(axis=1).to_numpy(copy=True)
transition = transition | table.ts_utc.diff().ne(pd.Timedelta(minutes=5)).to_numpy()
prefix = np.r_[0, np.cumsum(transition)]
assert (prefix[target + 1] - prefix[start + 1] == 0).all()
valid_targets = table[[c + '__valid' for c in TARGETS]].all(axis=1).to_numpy()
invalid_prefix = np.r_[0, np.cumsum(~valid_targets)]
assert (invalid_prefix[origin + 1] - invalid_prefix[start] == 0).all() and valid_targets[target].all()
assert np.array_equal(table.ts_utc.to_numpy()[origin], windows.origin_ts_utc.to_numpy())
assert np.array_equal(table.ts_utc.to_numpy()[target], windows.target_ts_utc.to_numpy())
assert np.array_equal(table.vision_usable.to_numpy()[origin], windows.vision_usable_at_origin.to_numpy())
# Drop test/embargo rows entirely from the modeling table; preserve row_id as the lookup index.
development = table[table['split'].isin(['train', 'validation'])].set_index('row_id').copy()
del table
assert not development['split'].eq('test').any()
save_json(ROOT / 'reports/window_audit.json', dict(upstream_window_rows=all_window_count,
    development_windows=len(windows), test_windows_used=0, all_intervals_and_target_qc_verified=True))
display(windows.groupby(['protocol', 'split', 'horizon_minutes']).size().unstack('horizon_minutes'))

## Cell 4 — Build past-only summaries and the seasonal reference

For each input, keep its latest value, 12-reading mean, 72-reading mean, and matching validity/coverage indicators. Add four past lags of each target. Calendar features at the forecast time are known in advance. Missing inputs become normalized zero only with their mask or coverage indicator.

In [ ]:
def build_history_features(protocol):
    """Summarize each continuous development group without mixing splits or compartments."""
    fields = upstream_scaler['protocols'][protocol]['history_features']
    stats = upstream_scaler['protocols'][protocol]['statistics']
    output = []
    subset = development[development.protocol == protocol]
    for _, group in subset.groupby(['experiment_id', 'compartment_id', 'split'], sort=True):
        group = group.sort_values('ts_utc')
        assert group.ts_utc.diff().dropna().eq(pd.Timedelta(minutes=5)).all()
        block = {}  # Collect columns first; avoid repeatedly fragmenting a wide DataFrame.
        for field in fields:
            valid = group[field + '__valid'].astype(bool) & np.isfinite(group[field])
            normalized = ((group[field] - stats[field]['mean']) / stats[field]['scale']).where(valid)
            block[field + '__latest'] = normalized.fillna(0)
            block[field + '__latest_valid'] = valid.astype(float)
            for length in [12, 72]:
                rolling = normalized.rolling(length, min_periods=1)
                block[field + f'__mean_{length}'] = rolling.mean().fillna(0)
                block[field + f'__coverage_{length}'] = valid.astype(float).rolling(length, min_periods=1).sum() / length
        for field in TARGETS:
            valid = group[field + '__valid'].astype(bool)
            normalized = ((group[field] - stats[field]['mean']) / stats[field]['scale']).where(valid)
            for lag in [6, 12, 36, 71]:
                block[field + f'__lag_{lag}'] = normalized.shift(lag).fillna(0)
                block[field + f'__lag_{lag}_valid'] = valid.shift(lag, fill_value=False).astype(float)
        output.append(pd.DataFrame(block, index=group.index).astype(np.float32))
    result = pd.concat(output).sort_index()
    assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
    return result

def design_matrix(features, rows):
    """Read only origin/history features and add a known future clock time."""
    values = features.loc[rows.origin_row].to_numpy(dtype=np.float64, copy=True)
    future_clock = rows.origin_ts_utc + pd.to_timedelta(rows.horizon_minutes, unit='min')
    minute = future_clock.dt.hour.to_numpy() * 60 + future_clock.dt.minute.to_numpy()
    angle = 2 * np.pi * minute / 1440
    values = np.column_stack([values, np.sin(angle), np.cos(angle)])
    return values, list(features.columns) + ['target_clock_sin', 'target_clock_cos']

def fit_seasonal_profile(protocol):
    """Fit pooled five-minute UTC clock means using valid training observations only."""
    training = development[(development.protocol == protocol) & (development['split'] == 'train')]
    slot = (training.ts_utc.dt.hour * 60 + training.ts_utc.dt.minute) // 5
    profile = np.zeros((288, len(TARGETS)), dtype=np.float64)
    for j, field in enumerate(TARGETS):
        values = training[field].where(training[field + '__valid'])
        means = values.groupby(slot).mean().reindex(range(288))
        profile[:, j] = means.fillna(values.mean()).to_numpy()
    assert np.isfinite(profile).all()
    return profile

def seasonal_predictions(profile, rows):
    """Look up a training-season average, never a yesterday reading or future measurement."""
    future_clock = rows.origin_ts_utc + pd.to_timedelta(rows.horizon_minutes, unit='min')
    slot = ((future_clock.dt.hour * 60 + future_clock.dt.minute) // 5).to_numpy(int)
    return profile[slot]

BASELINE_CONTRACT = dict(
    persistence='repeat the current valid target reading',
    seasonal_mean_train_only='pooled training-only mean at target UTC five-minute clock slot; missing slots use the training mean',
    daily_seasonal_persistence=dict(status='NOT_RUN_CONTRACT_CONFLICT',
        reason='target minus 24 hours is 18–23.75 hours before origin, outside the 72-reading history',
        no_silent_replacement=True),
    learned_models='Ridge and histogram gradient boosting predict change from current target, using the same history summaries and clock features',
    future_measured_covariates_used=False, vision_inputs_used=False, historical_prototypes_reused=False)
save_json(ROOT / 'data/baseline_contract.json', BASELINE_CONTRACT)
print('Daily seasonal persistence is unavailable under the frozen contract; separately named seasonal average is included.')

## Cell 5 — Fit and freeze Ridge and boosted models

Both models predict how much the target changes from its current value. Ridge chooses its penalty on development validation. Boosting uses one fixed configuration and no random internal validation split. Exported models are frozen before comparison reports are built; test labels are never consulted.

In [ ]:
metric_rows, selection_rows, prediction_parts, model_files, freeze_records = [], [], [], [], []
run_spec = dict(upstream_manifest_sha256=upstream_hash, versions=versions, seed=SEED, alphas=ALPHAS,
    boosting=HGB_SETTINGS, features='past-summaries-and-target-clock-v1', targets=TARGETS)

def record(protocol, horizon, model_name, rows, actual, predicted, persistence):
    """Store comparable predictions and physical-unit errors for exactly the same rows."""
    assert actual.shape == predicted.shape == persistence.shape == (len(rows), len(TARGETS))
    assert np.isfinite(predicted).all()
    for j, field in enumerate(TARGETS):
        error = predicted[:, j] - actual[:, j]
        persistence_mae = float(np.abs(persistence[:, j] - actual[:, j]).mean())
        mae = float(np.abs(error).mean())
        metric_rows.append(dict(protocol=protocol, horizon_minutes=horizon, target=field,
            error_unit=ERROR_UNITS[j], model=model_name, split='validation', n=len(rows),
            mae=mae, rmse=float(np.sqrt(np.square(error).mean())), bias=float(error.mean()),
            skill_vs_persistence=1 - mae / persistence_mae if persistence_mae > 0 else None))
        part = rows[['window_id', 'experiment_id', 'compartment_id', 'origin_ts_utc', 'target_ts_utc',
                     'vision_usable_at_origin']].copy()
        part['protocol'], part['horizon_minutes'], part['split'] = protocol, horizon, 'validation'
        part['target'], part['model'] = field, model_name
        part['actual'], part['predicted'] = actual[:, j], predicted[:, j]
        part['persistence_prediction'] = persistence[:, j]
        prediction_parts.append(part)

for protocol in PROTOCOLS:
    features = build_history_features(protocol)
    profile = fit_seasonal_profile(protocol)
    profile_path = f'models/{protocol}_seasonal_profile.json'
    save_json(ROOT / profile_path, dict(protocol=protocol, targets=TARGETS, utc_slot_minutes=5, means=profile.tolist()))
    model_files.append(profile_path)
    freeze_records.append(dict(path=profile_path, sha256=sha((ROOT / profile_path).read_bytes()), test_scored=False))
    target_scale = np.array([upstream_scaler['protocols'][protocol]['statistics'][t]['scale'] for t in TARGETS])
    for horizon in HORIZONS:
        subset = windows[(windows.protocol == protocol) & (windows.horizon_minutes == horizon)]
        training = subset[subset['split'] == 'train'].copy()
        validation = subset[subset['split'] == 'validation'].copy()
        assert len(training) and len(validation)
        x_train, feature_names = design_matrix(features, training)
        x_validation, validation_names = design_matrix(features, validation)
        assert feature_names == validation_names
        # This second scaler normalizes engineered summaries; its fit still sees training rows only.
        x_scaler = StandardScaler().fit(x_train)
        x_train = x_scaler.transform(x_train)
        x_validation = x_scaler.transform(x_validation)
        current_train = development.loc[training.origin_row, TARGETS].to_numpy(float)
        current_validation = development.loc[validation.origin_row, TARGETS].to_numpy(float)
        y_train = development.loc[training.target_row, TARGETS].to_numpy(float)
        y_validation = development.loc[validation.target_row, TARGETS].to_numpy(float)
        delta_train = (y_train - current_train) / target_scale
        assert np.isfinite(x_train).all() and np.isfinite(x_validation).all() and np.isfinite(delta_train).all()
        # Cache identity includes data contract, ordered row IDs, feature names, settings and versions.
        block_spec = dict(run_spec, protocol=protocol, horizon=horizon, feature_names=feature_names,
            train_windows_sha256=sha(training.window_id.to_numpy('<i8').tobytes()),
            validation_windows_sha256=sha(validation.window_id.to_numpy('<i8').tobytes()))
        fingerprint = sha(json.dumps(block_spec, sort_keys=True).encode())
        cache_path = CACHE / f'{protocol}_{horizon}_{fingerprint[:16]}.joblib'
        receipt_path = cache_path.with_suffix('.json')
        reusable = cache_path.exists() and receipt_path.exists()
        if reusable:
            receipt = json.loads(receipt_path.read_text())
            reusable = receipt.get('fingerprint') == fingerprint and receipt.get('sha256') == sha(cache_path.read_bytes())
        if reusable:
            bundle = joblib.load(cache_path)  # Only this notebook's fingerprinted local cache.
            print('Reused completed models:', protocol, horizon, flush=True)
        else:
            ridge_models, candidate_scores = [], []
            with threadpool_limits(limits=2):
                for alpha in ALPHAS:
                    fitted = Ridge(alpha=alpha, solver='cholesky').fit(x_train, delta_train)
                    pred = current_validation + fitted.predict(x_validation) * target_scale
                    ridge_models.append(fitted)
                    candidate_scores.append(np.abs(pred - y_validation).mean(axis=0))
                scores = np.stack(candidate_scores)
                best = scores.argmin(axis=0)
                coefficients = np.stack([ridge_models[k].coef_[j] for j, k in enumerate(best)])
                intercept = np.array([ridge_models[k].intercept_[j] for j, k in enumerate(best)])
                boosting = []
                for j, field in enumerate(TARGETS):
                    fitted = HistGradientBoostingRegressor(**HGB_SETTINGS).fit(x_train, delta_train[:, j])
                    boosting.append(fitted)
                    print('Trained boosting:', protocol, horizon, field, flush=True)
            bundle = dict(fingerprint=fingerprint, ridge_coef=coefficients, ridge_intercept=intercept,
                ridge_alpha=[ALPHAS[k] for k in best], candidate_scores=scores, boosting=boosting)
            temporary = cache_path.with_suffix('.tmp')
            joblib.dump(bundle, temporary, compress=3)
            temporary.replace(cache_path)
            save_json(receipt_path, dict(fingerprint=fingerprint, sha256=sha(cache_path.read_bytes())))
        assert bundle['fingerprint'] == fingerprint
        for k, alpha in enumerate(ALPHAS):
            for j, field in enumerate(TARGETS):
                selection_rows.append(dict(protocol=protocol, horizon_minutes=horizon, target=field,
                    alpha=alpha, validation_mae=float(bundle['candidate_scores'][k, j]),
                    selected=bool(alpha == bundle['ridge_alpha'][j])))
        with threadpool_limits(limits=2):
            ridge_prediction = current_validation + (x_validation @ bundle['ridge_coef'].T + bundle['ridge_intercept']) * target_scale
            boosting_prediction = current_validation + np.column_stack([m.predict(x_validation) for m in bundle['boosting']]) * target_scale
        model_config = dict(protocol=protocol, horizon_minutes=horizon, targets=TARGETS, features=feature_names,
            x_mean=x_scaler.mean_.tolist(), x_scale=x_scaler.scale_.tolist(), target_scale=target_scale.tolist(),
            prediction='current target + predicted normalized change * target_scale',
            ridge_alpha=bundle['ridge_alpha'], ridge_coef=bundle['ridge_coef'].tolist(),
            ridge_intercept=bundle['ridge_intercept'].tolist(), boosting_settings=HGB_SETTINGS,
            training_window_count=len(training), validation_window_count=len(validation),
            fingerprint=fingerprint, upstream_manifest_sha256=upstream_hash)
        stem = f'models/{protocol}_{horizon}min'
        save_json(ROOT / (stem + '.json'), model_config)
        joblib.dump(bundle['boosting'], ROOT / (stem + '_boosting.joblib'), compress=3)
        # Reload only our own new exports, then check prediction parity on a small validation slice.
        reloaded_config = json.loads((ROOT / (stem + '.json')).read_text())
        ridge_reload = current_validation[:32] + (x_validation[:32] @ np.array(reloaded_config['ridge_coef']).T
            + np.array(reloaded_config['ridge_intercept'])) * target_scale
        reloaded_boosting = joblib.load(ROOT / (stem + '_boosting.joblib'))
        with threadpool_limits(limits=2):
            boosting_reload = current_validation[:32] + np.column_stack([m.predict(x_validation[:32]) for m in reloaded_boosting]) * target_scale
        assert np.allclose(ridge_reload, ridge_prediction[:32], rtol=1e-8, atol=1e-8)
        assert np.allclose(boosting_reload, boosting_prediction[:32], rtol=1e-8, atol=1e-8)
        for relative in [stem + '.json', stem + '_boosting.joblib']:
            model_files.append(relative)
            freeze_records.append(dict(path=relative, sha256=sha((ROOT / relative).read_bytes()), test_scored=False))
        for model_name, prediction in [('persistence', current_validation),
                ('seasonal_mean_train_only', seasonal_predictions(profile, validation)),
                ('ridge', ridge_prediction), ('hist_gradient_boosting', boosting_prediction)]:
            record(protocol, horizon, model_name, validation, y_validation, prediction, current_validation)
        print('Completed:', protocol, horizon, 'minutes | train', len(training), '| validation', len(validation), flush=True)
        del bundle, reloaded_boosting, x_train, x_validation
        gc.collect()
    del features
    gc.collect()
predictions = pd.concat(prediction_parts, ignore_index=True)
del prediction_parts
metrics = pd.DataFrame(metric_rows)
pd.DataFrame(selection_rows).to_csv(ROOT / 'reports/ridge_selection.csv', index=False)
save_json(ROOT / 'models/freeze_record.json', freeze_records)
print('All models frozen; validation predictions:', len(predictions))

## Cell 6 — Matched metrics and day-block uncertainty

MAE is average absolute error; RMSE gives large errors more weight; bias shows systematic over- or under-prediction. A positive skill score beats persistence. We resample whole UTC days to keep nearby errors together. These are exploratory validation intervals, not calibrated prediction intervals or untouched-test claims.

In [ ]:
models = ['persistence', 'seasonal_mean_train_only', 'ridge', 'hist_gradient_boosting']
assert len(metrics) == len(PROTOCOLS) * len(HORIZONS) * len(TARGETS) * len(models)
assert predictions['split'].eq('validation').all()
assert not predictions.duplicated(['window_id', 'target', 'model']).any()
assert predictions.groupby(['window_id', 'target']).model.nunique().eq(len(models)).all()
assert predictions.groupby(['protocol', 'horizon_minutes', 'target', 'model']).size().sort_index().equals(
    metrics.set_index(['protocol', 'horizon_minutes', 'target', 'model']).n.sort_index())

intervals, compartment_metrics = [], []
for (protocol, horizon, target_name, model), group in predictions.groupby(
        ['protocol', 'horizon_minutes', 'target', 'model'], sort=True):
    error = group.predicted.to_numpy() - group.actual.to_numpy()
    base_error = group.persistence_prediction.to_numpy() - group.actual.to_numpy()
    differences = np.abs(error) - np.abs(base_error)
    day_sums = pd.DataFrame(dict(day=group.origin_ts_utc.dt.floor('D').to_numpy(), difference=differences))
    blocks = day_sums.groupby('day').difference.agg(['sum', 'count'])
    assert len(blocks) >= 2, 'Too few distinct days for a day-block interval.'
    rng = np.random.default_rng(SEED)
    sampled = rng.integers(0, len(blocks), size=(BOOTSTRAP_REPEATS, len(blocks)))
    estimates = blocks['sum'].to_numpy()[sampled].sum(axis=1) / blocks['count'].to_numpy()[sampled].sum(axis=1)
    low, high = np.quantile(estimates, [0.025, 0.975])
    intervals.append(dict(protocol=protocol, horizon_minutes=int(horizon), target=target_name, model=model,
        mae_difference_vs_persistence=float(differences.mean()), utc_day_blocks=len(blocks),
        exploratory_95_low=float(low), exploratory_95_high=float(high),
        interpretation='negative favors model; validation/model-selection and multiple comparisons limit inference'))
    for compartment, subgroup in group.groupby('compartment_id'):
        errors = subgroup.predicted.to_numpy() - subgroup.actual.to_numpy()
        compartment_metrics.append(dict(protocol=protocol, horizon_minutes=int(horizon), target=target_name,
            model=model, compartment_id=compartment, n=len(subgroup), mae=float(np.abs(errors).mean()),
            rmse=float(np.sqrt(np.square(errors).mean())), bias=float(errors.mean())))
pd.DataFrame(intervals).to_csv(ROOT / 'reports/paired_day_intervals.csv', index=False)
pd.DataFrame(compartment_metrics).to_csv(ROOT / 'reports/compartment_metrics.csv', index=False)
metrics.to_csv(ROOT / 'reports/metrics.csv', index=False)
predictions.to_parquet(ROOT / 'data/baseline_predictions.parquet', index=False)
best = metrics.loc[metrics.groupby(['protocol', 'horizon_minutes', 'target']).mae.idxmin()].copy()
best.to_csv(ROOT / 'reports/validation_best_model.csv', index=False)
metric_records = [{key: None if pd.isna(value) else value for key, value in row.items()}
                  for row in metrics.to_dict(orient='records')]
save_json(ROOT / 'reports/metrics.json', dict(split='validation', models=models,
    records=metric_records,
    daily_seasonal_persistence=BASELINE_CONTRACT['daily_seasonal_persistence'],
    best_model_rule='lowest development validation MAE; no test confirmation',
    uncertainty='paired UTC-day bootstrap of MAE differences; exploratory, not calibrated prediction coverage'))
display(metrics[metrics.horizon_minutes == 60].pivot(index=['protocol', 'target'], columns='model', values='mae'))

## Cell 7 — Plot comparisons and preserve the model-input contract

The plots show all five horizons, so a model cannot look good just because we displayed its best case. The saved contract lists exactly which inputs each exported model used and which data it did not use.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, protocol in enumerate(PROTOCOLS):
    for column, target_name in enumerate(TARGETS):
        axis = axes[row, column]
        selected = metrics[(metrics.protocol == protocol) & (metrics.target == target_name)]
        for model in models:
            curve = selected[selected.model == model].sort_values('horizon_minutes')
            axis.plot(curve.horizon_minutes, curve.mae, marker='o', label=model)
        axis.set_title(protocol + ' / ' + target_name)
        axis.set_xlabel('Forecast horizon (minutes)')
        axis.set_ylabel('Validation MAE (' + ERROR_UNITS[column] + ')')
axes[0, 0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(ROOT / 'reports/validation_horizons.png', dpi=130)
plt.show()
plt.close(fig)
input_contract = dict(schema_version='1.0', source='EDEN-07', targets=TARGETS, horizon_minutes=HORIZONS,
    inputs='same recorded climate/weather/control histories and target clock information for Ridge/boosting',
    history_steps=72, timestep_minutes=5, latest_to_oldest_minutes=355,
    summaries=['latest and valid flag', 'last 12 valid-value mean and coverage', 'last 72 valid-value mean and coverage'],
    target_lag_steps=[6, 12, 36, 71], known_future_features=['target UTC clock sine', 'target UTC clock cosine'],
    feature_order='listed separately in each models/<protocol>_<horizon>min.json',
    normalization='EDEN-07 training-only point scaler, then per-model StandardScaler fit on training window summaries',
    missing_inputs='normalized zero plus explicit validity/coverage; raw observations never imputed in source table',
    future_weather_or_actions_used=False, vision_used=False, test_used=False,
    target_model='learn normalized change from current reading; restore original units without clipping',
    evaluation='all four baselines scored on identical eligible validation rows per protocol/target/horizon',
    next='original EDEN-09 physics calibration; compare later models against these matched baselines')
save_json(ROOT / 'data/model_input_contract.json', input_contract)
save_json(ROOT / 'reports/coverage.json', dict(
    validation_windows_by_horizon=windows[windows['split']=='validation'].groupby(['protocol','horizon_minutes'])
        .agg(windows=('window_id','size'), with_image=('vision_usable_at_origin','sum')).reset_index().to_dict(orient='records'),
    vision_not_used_in_these_models=True, no_new_performance_claim_for_EDEN06_transfer=True))

## Cell 8 — Acceptance and named output ZIP

Success means a reproducible baseline comparison was produced. It does not require complex models to win. Daily seasonal persistence remains explicitly unavailable under the existing contract; test evaluation stays reserved for the later frozen evaluation stage.

In [ ]:
assert predictions['split'].eq('validation').all() and np.isfinite(predictions[['actual','predicted']].to_numpy()).all()
assert np.isfinite(metrics[['mae','rmse','bias']].to_numpy()).all()
for item in freeze_records:
    assert sha((ROOT / item['path']).read_bytes()) == item['sha256']
saved_predictions = pd.read_parquet(ROOT / 'data/baseline_predictions.parquet')
assert len(saved_predictions) == len(predictions) and saved_predictions['split'].eq('validation').all()
status = 'PASS_BASELINE_VALIDATION_SEASONAL_PERSISTENCE_UNAVAILABLE'
report = dict(status=status, protocols=PROTOCOLS, targets=TARGETS, horizons_minutes=HORIZONS,
    compared_models=models, validation_prediction_rows=len(predictions), metrics=len(metrics),
    test_scored=False, model_selection_split='validation', input_manifest_sha256=upstream_hash,
    full_training_data_used=True, future_measured_weather_or_controls_used=False, vision_inputs_used=False,
    daily_seasonal_persistence=BASELINE_CONTRACT['daily_seasonal_persistence'],
    limitations=['Validation is used for Ridge selection; reported comparisons are development evidence.',
        'Earlier prototype test summaries were inspected; do not claim completely untouched project test data.',
        'UTC-day intervals are exploratory under temporal dependence and multiple comparisons.',
        'Forecasts use recorded history, not known future action/weather scenarios.',
        'Daily seasonal persistence needs a separately approved longer-history contract; no such change is made.',
        'Vision remains sparse and canopy transfer unvalidated; its benefit is not tested here.'],
    next_notebook='EDEN-09 physics calibration, after review of these actual results')
save_json(ROOT / 'reports/baseline_report.json', report)
(ROOT / 'LICENSES.txt').write_text(
    'Derived from the 2024 Autonomous Greenhouse Challenge dataset, Wageningen University & Research / 4TU.ResearchData, CC BY 4.0.\n'
    'DOI: 10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1\n'
    'Scikit-learn model exports require the recorded library versions. Only load trusted joblib files.\n', encoding='utf-8')
required = ['data/upstream_windows.json','data/upstream_scaler.json','data/upstream_feature_contract.json','data/upstream_splits.json',
    'data/baseline_contract.json','data/baseline_predictions.parquet','data/model_input_contract.json',
    'reports/runtime.json','reports/input_verification.json','reports/window_audit.json','reports/ridge_selection.csv',
    'reports/paired_day_intervals.csv','reports/compartment_metrics.csv','reports/metrics.csv','reports/metrics.json',
    'reports/validation_best_model.csv','reports/validation_horizons.png','reports/coverage.json','reports/baseline_report.json',
    'models/freeze_record.json','LICENSES.txt'] + model_files
assert len(required) == len(set(required))
file_manifest = []
for relative in required:
    payload = (ROOT / relative).read_bytes()
    file_manifest.append(dict(path=relative, size=len(payload), sha256=sha(payload)))
manifest = dict(schema_version='1.0', producer='EDEN-08', code_version='1.0', status=status,
    generated_utc=datetime.now(timezone.utc).isoformat(), upstream_manifest_sha256=upstream_hash,
    upstream_producer='EDEN-07', test_scored=False, files=file_manifest)
save_json(ROOT / 'manifest.json', manifest)
with zipfile.ZipFile(RESULT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as bundle:
    for relative in required + ['manifest.json']:
        bundle.write(ROOT / relative, 'eden/' + relative)
with zipfile.ZipFile(RESULT) as bundle:
    assert bundle.testzip() is None
    for item in file_manifest:
        payload = bundle.read('eden/' + item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256']
print(status)
print('Return:', RESULT, '| bytes:', RESULT.stat().st_size)

### If a cell stops

- **EDEN-07 not found:** attach its result ZIP or extracted folder, keeping one distinct run.
- **Interrupted training:** rerun all cells in the same session. Completed protocol/horizon model blocks are verified and reused; an unfinished block is retrained.
- **Hash/split/row-pointer mismatch:** return the error. Do not remove the check or mix result versions.
- **Boosting takes time:** it trains 30 CPU models. This is the full five-horizon comparison, not a demonstration subset.
- **Persistence wins:** keep that result. A stronger model has to earn its place through matched evidence.

Return only the named EDEN-08 results ZIP. No local notebook execution or training was performed while preparing this file.